# Notebook 006. Study-area and reference-label statistics
-------

Quantitative analysis of the area of interest (AOI) and of the reference labels:
the numbers quoted in the manuscript's study-area and reference-label sections, and
the supplementary figure of baseline predictors by class and CORINE forest type:

1. Area of interest: the two Natura 2000 sites, their extent and area.
2. Altitude and slope distributions across the AOI (FABDEM).
3. Forest-type composition (CORINE 311 / 312 / 313).
4. Reference labels and parcel-level baseline-predictor statistics.
5. Study-area statistics for the manuscript: CORINE land-cover classes, elevation and
   the parcel-map domain.
6. Class summary, parcel areas and class by forest type.
7. Baseline predictors by class (table and figure).
8. Species composition.
9. Stand age, ownership and forest type.
10. Forest type: species attribute versus CORINE, with the dominance-threshold sensitivity.
11. Baseline predictors by class and forest type (species attribute, then the CORINE
    version used in the manuscript).

Sections 8 to 11 read the forest-record attributes (species composition, stand age,
ownership). Labels rebuilt from the published dataset (see notebook 003) do not contain those columns, so only sections 1 to 7 and the CORINE figure of section 11 are
meaningful in that case.

## 1. Area of interest

The AOI is the two-site Natura 2000 boundary: `ROSAC0122` Munții Făgăraș and
`ROSAC0381` Râul Târgului – Argeșel – Râușor. Areas are measured on the equal-area
project CRS, so the union area is in true hectares.

In [ ]:
# Setup and AOI extent / area. Loads the AOI as individual sites (for the per-site
# breakdown) and dissolved (for the union area and grid masking). EPSG:3035 is
# equal-area, so areas are true hectares.
import logging
import os
import zipfile

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from affine import Affine
from matplotlib.colors import LightSource, to_rgba
from rasterio.enums import Resampling
from rasterio.io import MemoryFile
from rasterio.mask import mask as rio_mask

from utils import raster_io, terminology, vector_io
from utils.paths import get_project_paths
from utils.style import get_figure_size, save_figure, use_publication_style
from utils.vector_io import load_aoi, repair_geometries

use_publication_style()  # Charis SIL publication typeface (utils.style)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

paths = get_project_paths()
NOTEBOOK = "006_aoi_and_reference_label_statistics"
NODATA = terminology.NODATA
RES = terminology.REF_RESOLUTION_M
N_THREADS = os.cpu_count()
PIXEL_HA = (RES * RES) / 1e4  # 0.01 ha per 10 m pixel

aoi_sites = load_aoi()  # two features, EPSG:3035
aoi = load_aoi(dissolve=True)  # single-feature union
grid = raster_io.open_reference_grid()
AOI_AREA_HA = vector_io.unique_area_ha(aoi)

# AOI footprint on the reference grid (used for percentage denominators below).
aoi_mask = raster_io.rasterize_mask(aoi.geometry, grid, all_touched=True).astype(bool)
AOI_PIXELS = int(aoi_mask.sum())

site_areas = (
    aoi_sites.assign(area_ha=aoi_sites.geometry.area / 1e4)[["SITECODE", "SITENAME", "area_ha"]]
    .sort_values("area_ha", ascending=False)
    .reset_index(drop=True)
)
minx, miny, maxx, maxy = aoi.total_bounds
print(f"[aoi] {len(aoi_sites)} Natura 2000 sites; union area {AOI_AREA_HA:,.0f} ha")
print(
    f"[aoi] extent EPSG:3035 ({minx:,.0f}, {miny:,.0f}) - ({maxx:,.0f}, {maxy:,.0f}) m; "
    f"{(maxx - minx) / 1e3:.1f} x {(maxy - miny) / 1e3:.1f} km"
)
print(f"[aoi] reference-grid footprint: {AOI_PIXELS:,} px (~{AOI_PIXELS * PIXEL_HA:,.0f} ha)")
site_areas.round(0)

## 2. Altitude and slope

Distributions across the AOI from the processed FABDEM elevation and slope layers
(10 m, EPSG:3035), over the valid (in-AOI) pixels. The dashed line marks the median.

In [ ]:
# Altitude and slope distributions from the processed FABDEM layers.
ELEV_PATH = paths.processed / "rasters" / "fabdem_10m" / "elevation_3035_10m.tif"
SLOPE_PATH = paths.processed / "rasters" / "fabdem_10m" / "slope_deg_3035_10m.tif"


def masked_values(path):
    with rasterio.open(path) as src:
        return src.read(1, masked=True).compressed()


elev = masked_values(ELEV_PATH)
slope = masked_values(SLOPE_PATH)


def describe(values, name, unit):
    p05, p25, p50, p75, p95 = np.percentile(values, [5, 25, 50, 75, 95])
    return {
        "variable": name,
        "unit": unit,
        "n_pixels": int(values.size),
        "min": float(values.min()),
        "p05": float(p05),
        "p25": float(p25),
        "median": float(p50),
        "mean": float(values.mean()),
        "p75": float(p75),
        "p95": float(p95),
        "max": float(values.max()),
    }


terrain_stats = pd.DataFrame(
    [describe(elev, "elevation", "m"), describe(slope, "slope", "degrees")]
)
print(
    f"[terrain] elevation {elev.min():.0f}-{elev.max():.0f} m (median {np.median(elev):.0f}); "
    f"slope {slope.min():.0f}-{slope.max():.0f} deg (median {np.median(slope):.0f})"
)

fig, axes = plt.subplots(1, 2, figsize=get_figure_size("double", aspect=0.42))
axes[0].hist(
    elev, bins=60, color=terminology.PALETTE_CATEGORICAL["blue"], edgecolor="white", linewidth=0.2
)
axes[0].axvline(np.median(elev), color="black", lw=1, ls="--")
axes[0].set_xlabel("Elevation (m)")
axes[0].set_ylabel("Pixel count")
axes[1].hist(
    slope,
    bins=60,
    color=terminology.PALETTE_CATEGORICAL["magenta"],
    edgecolor="white",
    linewidth=0.2,
)
axes[1].axvline(np.median(slope), color="black", lw=1, ls="--")
axes[1].set_xlabel("Slope (degrees)")
axes[1].set_ylabel("Pixel count")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
fig.tight_layout()
save_figure(fig, f"{NOTEBOOK}/altitude_slope_distributions", data=terrain_stats)
plt.show()
terrain_stats.round(1)

## 3. Forest-type composition

CORINE Land Cover forest classes within the AOI (`utils.terminology.CORINE_FOREST_CLASSES`):
broad-leaved (311), coniferous (312) and mixed (313). Areas are recomputed from the
AOI-clipped geometry. Percentages are quoted against the total mapped forest and
against the whole AOI.

In [ ]:
# Forest-type composition and per-type elevation from CORINE within the AOI.
CORINE_VEC = paths.processed / "vectors" / "corine_land_cover" / "corine_forest_aoi_3035.gpkg"
ELEV_PATH = paths.processed / "rasters" / "fabdem_10m" / "elevation_3035_10m.tif"
corine = gpd.read_file(CORINE_VEC)
forest_order = list(terminology.CORINE_FOREST_CLASSES.values())  # broadleaf, coniferous, mixed

# Read the elevation grid once for per-forest-type zonal statistics (pixels whose
# centre falls in a forest polygon of that type).
with rasterio.open(ELEV_PATH) as src:
    elev_grid = src.read(1, masked=True)
elev_valid = ~np.ma.getmaskarray(elev_grid)

rows = []
for ftype in forest_order:
    polys = corine.loc[corine["forest_type"] == ftype, "geometry"]
    type_mask = raster_io.rasterize_mask(polys, grid, all_touched=False).astype(bool)
    ev = elev_grid.data[type_mask & elev_valid]
    rows.append(
        {
            "forest_type": ftype,
            "area_ha": float(polys.area.sum()) / 1e4,
            "elev_mean_m": float(ev.mean()),
            "elev_min_m": float(ev.min()),
            "elev_max_m": float(ev.max()),
        }
    )
forest_comp = pd.DataFrame(rows)
forest_total = float(forest_comp["area_ha"].sum())
forest_comp["pct_of_forest"] = 100 * forest_comp["area_ha"] / forest_total
forest_comp["pct_of_aoi"] = 100 * forest_comp["area_ha"] / AOI_AREA_HA
print(
    f"[corine] mapped forest within the AOI: {forest_total:,.0f} ha "
    f"({100 * forest_total / AOI_AREA_HA:.1f}% of the AOI)"
)
for r in rows:
    print(
        f"[corine] {r['forest_type']}: mean elevation {r['elev_mean_m']:.0f} m "
        f"(range {r['elev_min_m']:.0f}-{r['elev_max_m']:.0f} m)"
    )

# Land-cover categories; teal/orange are reserved for the OGF / non-OGF labels.
FOREST_TYPE_COLOURS = {
    "broadleaf": terminology.PALETTE_CATEGORICAL["light_green"],
    "coniferous": terminology.PALETTE_CATEGORICAL["blue"],
    "mixed": terminology.PALETTE_CATEGORICAL["yellow"],
}
fig, ax = plt.subplots(figsize=get_figure_size("single", aspect=0.72))
bars = ax.bar(
    forest_comp["forest_type"],
    forest_comp["area_ha"],
    color=[FOREST_TYPE_COLOURS[t] for t in forest_comp["forest_type"]],
    edgecolor="black",
    linewidth=0.4,
)
for bar, pct in zip(bars, forest_comp["pct_of_forest"], strict=True):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{pct:.0f}%",
        ha="center",
        va="bottom",
        fontsize=8,
    )
ax.set_ylabel("Area (ha)")
ax.set_xlabel("CORINE forest type")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_figure(fig, f"{NOTEBOOK}/forest_type_composition", data=forest_comp)
plt.show()
forest_comp[
    [
        "forest_type",
        "area_ha",
        "pct_of_forest",
        "pct_of_aoi",
        "elev_mean_m",
        "elev_min_m",
        "elev_max_m",
    ]
].round(1)

## 4. Reference labels and baseline-predictor statistics

Load the reference labels, compute each parcel's mean of the baseline predictors over the valid pixels of the baseline stack, and derive the reference group (OGF, Non-OGF, Unlabelled) from the class column.

In [ ]:
import logging

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.features import rasterize
from scipy import ndimage

from utils import terminology
from utils.paths import get_project_paths

logging.basicConfig(level=logging.INFO, force=True)

NODATA = terminology.NODATA
BASELINE_BANDS = terminology.FEATURE_SET_BANDS["baseline"]  # eight bands, fixed order
FOREST_AOI_HA = 152_366.0  # forested AOI; the percentage denominator used in the manuscript

paths = get_project_paths()
baseline_path = paths.processed / "rasters" / "stacks_10m" / "baseline_3035_10m.tif"
labels = gpd.read_file(paths.labels / "ogf_reference_labels.gpkg", layer="parcels")
labels["area_ha"] = labels.geometry.area / 1e4
labels["ogf_group"] = np.select(
    [labels["class"].str.startswith("OGF"), labels["class"].str.startswith("Non_OGF")],
    ["OGF", "Non-OGF"],
    default="Unlabelled",
)

# Read the full eight-band baseline stack into memory (~1.3 GB) for vectorised zonal means.
with rasterio.open(baseline_path) as src:
    stack = src.read().astype("float32")
    transform = src.transform
    out_shape = (src.height, src.width)

ids = np.arange(1, len(labels) + 1, dtype="int32")
parcel_raster = rasterize(
    zip(labels.geometry, ids, strict=True),
    out_shape=out_shape,
    transform=transform,
    fill=0,
    dtype="int32",
)
valid = (stack != NODATA).all(axis=0)  # one common footprint across all eight bands
parcel_valid = np.where(valid, parcel_raster, 0)
counts = np.bincount(parcel_valid.ravel(), minlength=len(labels) + 1)[1 : len(labels) + 1]

parcel_stats = labels[["parcel_id", "class", "ogf_group", "area_ha", "corine_forest_type"]].copy()
for index, band in enumerate(BASELINE_BANDS):
    means = ndimage.mean(stack[index], labels=parcel_valid, index=ids)
    parcel_stats[band] = np.where(counts > 0, means, np.nan)

del stack, valid, parcel_raster, parcel_valid  # release the raster-sized arrays

missing = int(parcel_stats[BASELINE_BANDS[0]].isna().sum())
for group in ["OGF", "Non-OGF"]:
    sub = parcel_stats[parcel_stats["ogf_group"].eq(group)]
    finite = int(sub[BASELINE_BANDS[0]].notna().sum())
    print(f"[zonal] {group}: {finite} of {len(sub)} parcels with baseline statistics")
print(
    f"[zonal] {len(BASELINE_BANDS)} bands; {missing} parcels outside the baseline footprint (NaN)"
)

## 5. Study-area statistics for the manuscript

Study-area extent and terrain: total area of the Natura 2000 AOI, its FABDEM elevation range, the area and elevation of every CORINE land-cover class within it, and the share of the forested AOI (CORINE forest classes) that the parcel-map domain covers.

In [ ]:
import pandas as pd

# AOI extent and CORINE land-cover classes: area and FABDEM elevation.
elevation_path = paths.processed / "rasters" / "fabdem_10m" / "elevation_3035_10m.tif"
# Full CORINE 2018 (all classes), clipped to the AOI below; the processed
# corine_forest_aoi_3035.gpkg holds only the three forest classes (311/312/313).
corine_path = paths.raw / "vectors" / "corine_land_cover" / "corine_clc2018_romania.gpkg"

aoi = gpd.read_file(paths.aoi)

with rasterio.open(elevation_path) as src:
    elevation = src.read(1)
    elev_transform = src.transform
    elev_shape = (src.height, src.width)
    elev_nodata = src.nodata
    raster_crs = src.crs

aoi = aoi.to_crs(raster_crs)
elev_valid = elevation != elev_nodata


def elevation_stats(mask):
    """min/max/mean/median elevation (m) over valid pixels inside a boolean mask."""
    values = elevation[mask & elev_valid]
    if values.size == 0:
        keys = ["elev_min_m", "elev_max_m", "elev_mean_m", "elev_median_m"]
        return pd.Series(dict.fromkeys(keys, np.nan))
    return pd.Series(
        {
            "elev_min_m": values.min(),
            "elev_max_m": values.max(),
            "elev_mean_m": values.mean(),
            "elev_median_m": np.median(values),
        }
    )


def footprint(geometries):
    """Boolean pixel mask on the elevation grid covered by the given geometries."""
    return rasterize(
        [(geom, 1) for geom in geometries],
        out_shape=elev_shape,
        transform=elev_transform,
        fill=0,
        dtype="uint8",
    ).astype(bool)


# Total AOI extent (all pixels enclosed by aoi_natura_2000.gpkg).
aoi_area_ha = float(aoi.geometry.area.sum() / 1e4)
aoi_elev = elevation_stats(footprint(aoi.geometry))

print("[section 2.2] AOI (aoi_natura_2000.gpkg):")
print(f"   area: {aoi_area_ha:,.1f} ha ({aoi_area_ha / 100:,.2f} km2)")
print(
    "   elevation (m): "
    f"min {aoi_elev['elev_min_m']:.1f}, max {aoi_elev['elev_max_m']:.1f}, "
    f"mean {aoi_elev['elev_mean_m']:.1f}, median {aoi_elev['elev_median_m']:.1f}"
)

# All CORINE land-cover classes within the AOI. Clip the national layer to the AOI,
# then burn class codes into one raster for vectorised per-class elevation.
corine = gpd.read_file(corine_path, bbox=tuple(aoi.total_bounds)).to_crs(raster_crs)
corine = gpd.clip(corine, aoi, keep_geom_type=True)
corine = corine[~corine.geometry.is_empty & corine.geometry.notna()].copy()
corine["clc_code"] = corine["Code_18"].astype(int)
corine["area_ha"] = corine.geometry.area / 1e4

code_grid = rasterize(
    zip(corine.geometry, corine["clc_code"], strict=True),
    out_shape=elev_shape,
    transform=elev_transform,
    fill=0,
    dtype="int32",
)

rows = []
for clc_code, group in corine.groupby("clc_code"):
    class_area_ha = float(group["area_ha"].sum())
    row = {
        "clc_code": clc_code,
        "class": terminology.CORINE_CLC_CLASSES.get(clc_code, str(clc_code)),
        "area_ha": class_area_ha,
        "area_km2": class_area_ha / 100.0,
        "pct_of_aoi": 100.0 * class_area_ha / aoi_area_ha,
    }
    row.update(elevation_stats(code_grid == clc_code))
    rows.append(row)

corine_table = (
    pd.DataFrame(rows).sort_values("area_ha", ascending=False).set_index(["clc_code", "class"])
)
corine_table.index.names = ["CLC code", "CORINE class"]

print("\n[section 2.2] CORINE land-cover classes within the AOI (area and elevation):")
print(corine_table.round(1).to_string())
print(
    f"[section 2.2] {len(corine_table)} classes; "
    f"total {corine_table['area_ha'].sum():,.1f} ha "
    f"({corine_table['pct_of_aoi'].sum():.1f}% of the AOI)"
)

# Proportion of the forested AOI (CORINE forest classes) that the parcel-map domain covers.
# The domain (parcel_map_clean.gpkg) is the area this study maps; some CORINE forest within
# the AOI lies outside it, and some mapped parcels sit on non-forest CORINE land.
forest = corine[corine["clc_code"].isin(list(terminology.CORINE_FOREST_CLASSES))].copy()
forest["geometry"] = forest.geometry.make_valid()
forest_geom = forest.union_all()
forest_ha = forest_geom.area / 1e4

domain = gpd.read_file(
    paths.processed / "vectors" / "forest_records" / "parcel_map_clean.gpkg"
).to_crs(raster_crs)
domain["geometry"] = domain.geometry.make_valid()
domain_geom = domain.union_all()
domain_ha = domain_geom.area / 1e4
overlap_ha = forest_geom.intersection(domain_geom).area / 1e4

print("\n[section 2.2] parcel-map domain vs the forested AOI (CORINE forest classes):")
print(
    f"   forested AOI {forest_ha:,.1f} ha; parcel-map domain {domain_ha:,.1f} ha; "
    f"overlap {overlap_ha:,.1f} ha"
)
print(
    f"   {100 * overlap_ha / forest_ha:.1f}% of the forested AOI is covered by the domain; "
    f"{100 * overlap_ha / domain_ha:.1f}% of the domain is CORINE forest"
)

## 6. Class summary

Count parcels and area per class and report old-growth prevalence by area and by parcel count.

In [ ]:
GROUP_ORDER = ["OGF", "Non-OGF", "Unlabelled"]

summary = (
    parcel_stats.groupby("ogf_group")
    .agg(parcels=("parcel_id", "size"), area_ha=("area_ha", "sum"))
    .reindex(GROUP_ORDER)
)
summary["pct_of_forest"] = 100.0 * summary["area_ha"] / FOREST_AOI_HA

labelled = summary.loc[["OGF", "Non-OGF"]]
prevalence_area = labelled.loc["OGF", "area_ha"] / labelled["area_ha"].sum()
prevalence_count = labelled.loc["OGF", "parcels"] / labelled["parcels"].sum()

print(summary.round(1).to_string())
print(f"[prevalence] area-based {prevalence_area:.3f}; parcel-count {prevalence_count:.3f}")

In [ ]:
import pandas as pd

# Parcel area (ha) distribution: all parcels and split by old-growth label
# (OGF true, Non-OGF false, Unlabelled blank).
AREA_GROUPS = {
    "All parcels": parcel_stats["area_ha"],
    "OGF (true)": parcel_stats.loc[parcel_stats["ogf_group"].eq("OGF"), "area_ha"],
    "Non-OGF (false)": parcel_stats.loc[parcel_stats["ogf_group"].eq("Non-OGF"), "area_ha"],
    "Unlabelled (blank)": parcel_stats.loc[parcel_stats["ogf_group"].eq("Unlabelled"), "area_ha"],
}


def area_summary(values):
    return pd.Series(
        {
            "n": values.size,
            "mean": values.mean(),
            "p05": values.quantile(0.05),
            "p50": values.quantile(0.50),
            "p95": values.quantile(0.95),
            "min": values.min(),
            "max": values.max(),
        }
    )


area_table = pd.DataFrame({name: area_summary(values) for name, values in AREA_GROUPS.items()})

print("[area statistics] parcel area (ha) by old-growth label:")
print(area_table.round(2).to_string())

# Parcels whose area rounds to 0.00 ha (< 0.005 ha); none are exactly zero.
ZERO_HA = 0.005
zero_counts = {name: int((values < ZERO_HA).sum()) for name, values in AREA_GROUPS.items()}
print(
    f"\n[area statistics] parcels rounding to 0.00 ha (< {ZERO_HA} ha): "
    + ", ".join(f"{name} {count}" for name, count in zero_counts.items())
)

In [ ]:
import pandas as pd

from utils import terminology

# Class x forest type: the species-attribute type is the class suffix, lower-cased.
FOREST_TYPES = list(terminology.CORINE_FOREST_CLASSES.values())  # broadleaf, coniferous, mixed
DISPLAY = {t: t.title() for t in FOREST_TYPES}

strata = parcel_stats[parcel_stats["ogf_group"].isin(["OGF", "Non-OGF"])].copy()
strata["forest_type"] = strata["class"].str.split("_").str[-1].str.lower()
strata = strata[strata["forest_type"].isin(FOREST_TYPES)]

index = pd.MultiIndex.from_product(
    [["OGF", "Non-OGF"], FOREST_TYPES], names=["class", "forest type"]
)
by_type = (
    strata.groupby(["ogf_group", "forest_type"])
    .agg(parcels=("parcel_id", "size"), area_ha=("area_ha", "sum"))
    .reindex(index)
    .fillna(0)
)
by_type["pct_of_forest"] = 100.0 * by_type["area_ha"] / FOREST_AOI_HA
by_type = by_type.rename(index=DISPLAY, level="forest type")

print("[class x forest type] parcels, area (ha) and percentage of forested AOI:")
print(by_type.round(1).to_string())

## 7. Baseline predictors by class

Tabulate and plot the eight baseline predictors as parcel-level medians and distributions by class.

In [ ]:
import pandas as pd

from utils import terminology

DISTANCE_BANDS = {"dist_paved_road_m", "dist_unpaved_road_m", "dist_footpath_m"}
GROUPS = ["OGF", "Non-OGF"]

rows = {}
for band in BASELINE_BANDS:
    label = terminology.BAND_LABELS[band]
    scale = 1000.0 if band in DISTANCE_BANDS else 1.0
    if band in DISTANCE_BANDS:
        label = label.replace("(m)", "(km)")
    row = {}
    for group in GROUPS:
        values = parcel_stats.loc[parcel_stats["ogf_group"].eq(group), band].dropna() / scale
        q1, q3 = values.quantile(0.25), values.quantile(0.75)
        row[group] = f"{values.median():,.1f} ({q1:,.1f}-{q3:,.1f})"
    rows[label] = row

predictor_table = pd.DataFrame(rows).T
predictor_table.index.name = "Predictor (median, Q1-Q3)"
predictor_table.columns = ["Old-growth", "Non-old-growth"]
print(predictor_table.to_string())

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from utils import terminology
from utils.style import get_figure_size, save_figure, use_publication_style

use_publication_style()  # Charis SIL publication typeface (utils.style)

NOTEBOOK = "006_aoi_and_reference_label_statistics"
GROUPS = ["OGF", "Non-OGF"]
GROUP_COLOURS = {
    "OGF": terminology.SEMANTIC_COLOURS["ogf"],
    "Non-OGF": terminology.SEMANTIC_COLOURS["non_ogf"],
}
GROUP_LABELS = {"OGF": "Old-growth", "Non-OGF": "Non-old-growth"}
X_LABELS = ["Old-\ngrowth", "Non-old-\ngrowth"]
DISTANCE_BANDS = {"dist_paved_road_m", "dist_unpaved_road_m", "dist_footpath_m"}
PANEL_TITLES = {
    "elevation_m": "Elevation (m)",
    "slope_deg": "Slope (degrees)",
    "heat_load_index": "Heat load index",
    "dist_paved_road_m": "Distance to paved\nroad (km)",
    "dist_unpaved_road_m": "Distance to unpaved\nroad (km)",
    "dist_footpath_m": "Distance to\nfootpath (km)",
    "bio01_mean_annual_temp_degC": "Mean annual\ntemperature (deg C)",
    "bio12_annual_precip_mm": "Annual precipitation\n(mm)",
}

n_by_group = {}
for group in GROUPS:
    finite = parcel_stats.loc[parcel_stats["ogf_group"].eq(group), BASELINE_BANDS[0]]
    n_by_group[group] = int(finite.notna().sum())

with plt.rc_context(
    {
        "font.size": 7,
        "axes.titlesize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
    }
):
    fig, axes = plt.subplots(2, 3, figsize=get_figure_size("double", aspect=0.62))
    for ax, band in zip(axes.flat, BASELINE_BANDS, strict=True):
        scale = 1000.0 if band in DISTANCE_BANDS else 1.0
        data = []
        for group in GROUPS:
            values = parcel_stats.loc[parcel_stats["ogf_group"].eq(group), band].dropna()
            data.append(values.to_numpy() / scale)
        box = ax.boxplot(
            data,
            widths=0.6,
            patch_artist=True,
            showfliers=False,
            medianprops=dict(color="black", linewidth=1.0),
        )
        for patch, group in zip(box["boxes"], GROUPS, strict=True):
            patch.set_facecolor(GROUP_COLOURS[group])
            patch.set_alpha(0.85)
        ax.set_title(PANEL_TITLES[band], pad=3)
        ax.set_xticks([1, 2])
        ax.set_xticklabels(X_LABELS)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.margins(y=0.05)

    handles = [
        Patch(
            facecolor=GROUP_COLOURS[group],
            edgecolor="black",
            alpha=0.85,
            label=f"{GROUP_LABELS[group]} (n = {n_by_group[group]:,})",
        )
        for group in GROUPS
    ]
    fig.legend(
        handles=handles, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.93)
    )
    fig.suptitle("Baseline predictors by reference class", y=0.99, fontsize=9)
    fig.subplots_adjust(left=0.07, right=0.99, top=0.83, bottom=0.13, wspace=0.42, hspace=0.68)

save_figure(
    fig,
    f"{NOTEBOOK}/baseline_predictors_by_class",
    data=parcel_stats.loc[parcel_stats["ogf_group"].isin(GROUPS), ["ogf_group", *BASELINE_BANDS]],
)
plt.show()

## 8. Species composition

Rank species by area-weighted presence within each class and report the dominant-species spruce-to-beech area ratio.

In [ ]:
import re

from utils import terminology

PAIR = re.compile(r"(\d{1,2})([A-Z]{2,4})")
TOP_N = 5
GROUPS = ["OGF", "Non-OGF"]


def species_fractions(composition):
    """Return {species_code: share} for a composition string, normalised to sum to one."""
    if not isinstance(composition, str):
        return {}
    pairs = PAIR.findall(composition)
    total = sum(int(share) for share, _ in pairs)
    return {code: int(share) / total for share, code in pairs} if total else {}


for group in GROUPS:
    sub = labels[labels["ogf_group"].eq(group)]
    area_by_species = {}
    for composition, area in zip(sub["final_composition"], sub["area_ha"], strict=True):
        for code, share in species_fractions(composition).items():
            area_by_species[code] = area_by_species.get(code, 0.0) + share * area
    composed = sum(area_by_species.values())
    class_area = sub["area_ha"].sum()
    missing_pct = 100.0 * (class_area - composed) / class_area
    print(
        f"\n[{group}] composed {composed:,.0f} ha of {class_area:,.0f} ha "
        f"({missing_pct:.1f}% without species)"
    )
    ranked = sorted(area_by_species.items(), key=lambda item: item[1], reverse=True)[:TOP_N]
    for code, area in ranked:
        species = terminology.SPECIES[code]
        name = species.english if species.latin is None else f"{species.english} ({species.latin})"
        print(f"   {name}: {area:,.0f} ha, {100 * area / composed:.1f}%")

print()
for group in GROUPS:
    dominant_area = (
        labels.loc[labels["ogf_group"].eq(group)].groupby("dominant_code")["area_ha"].sum()
    )
    spruce = dominant_area.get("MO", 0.0)
    beech = dominant_area.get("FA", 0.0)
    ratio = spruce / beech if beech else float("nan")
    print(f"[{group}] dominant spruce:beech area ratio = {ratio:.2f}")

## 9. Stand age, ownership and forest type

Summarise stand age, ownership (area in ha and parcel counts, each with the % of the group total) and forest type, for each reference group (OGF, Non-OGF, Unlabelled) and for all parcels combined. Parcels with no recorded owner are shown as "Unassigned"; forest type is derived from stand composition, so it also covers unlabelled parcels.

In [ ]:
import pandas as pd

from utils.forestry import classify_forest_type

# Columns: the three reference groups plus a combined "all parcels" total.
GROUP_COLUMNS = ["OGF", "Non-OGF", "Unlabelled"]
ALL_LABEL = "All parcels"
OWNER_COLUMNS = [*GROUP_COLUMNS, ALL_LABEL]


# Stand age (years), summarised per group and across all parcels with an age.
def age_summary(mask):
    age = labels.loc[mask, "final_age"].dropna()
    return pd.Series(
        {"n": age.size, "median": age.median(), "Q1": age.quantile(0.25), "Q3": age.quantile(0.75)}
    )


age_masks = {group: labels["ogf_group"].eq(group) for group in GROUP_COLUMNS}
age_masks[ALL_LABEL] = pd.Series(True, index=labels.index)
age_table = pd.DataFrame({group: age_summary(mask) for group, mask in age_masks.items()})
print("[age: stand age (years) by class]")
print(age_table.round(0).astype("Int64").to_string())

# Ownership tables. Parcels with no recorded owner are kept as an "Unassigned" row so
# each column's percentages are a share of that group's full total (they sum to 100%).
owner = labels.assign(ownership=labels["ownership_type"].fillna("Unassigned"))


def ownership_with_pct(pivot, value_label):
    """Add a per-column % of group total; keep Unassigned last, others by descending total."""
    pivot = pivot.reindex(columns=GROUP_COLUMNS).fillna(0)
    pivot[ALL_LABEL] = pivot.sum(axis=1)
    named = pivot.drop(index="Unassigned", errors="ignore").sort_values(ALL_LABEL, ascending=False)
    pivot = pivot.reindex([*list(named.index), "Unassigned"])
    table = pd.DataFrame(index=pivot.index)
    for column in OWNER_COLUMNS:
        table[(column, value_label)] = pivot[column].round(0).astype(int)
        table[(column, "%")] = (100.0 * pivot[column] / pivot[column].sum()).round(1)
    table.columns = pd.MultiIndex.from_tuples(table.columns)
    table.index.name = "ownership_type"
    return table


area_pivot = owner.pivot_table(
    index="ownership", columns="ogf_group", values="area_ha", aggfunc="sum"
)
ownership_area = ownership_with_pct(area_pivot, "ha")
print("\n[ownership: area (ha) and % of group total]")
print(ownership_area.to_string())

count_pivot = owner.pivot_table(
    index="ownership", columns="ogf_group", values="parcel_id", aggfunc="size"
)
ownership_count = ownership_with_pct(count_pivot, "n")
print("\n[ownership: parcel counts and % of group total]")
print(ownership_count.to_string())

# Forest type: species-attribute type derived from stand composition for every parcel.
# This reproduces the OGF/Non-OGF class suffix exactly (so those counts are unchanged)
# while also classifying the ~52% of unlabelled parcels that carry composition data;
# parcels without composition fall under "Unclassified".
FOREST_ORDER = ["Broadleaf", "Coniferous", "Mixed", "Unclassified"]
species_forest_type = (
    pd.Series(
        [
            classify_forest_type(b, c)
            for b, c in zip(labels["pct_broadleaf"], labels["pct_coniferous"], strict=False)
        ],
        index=labels.index,
    )
    .fillna("unclassified")
    .str.title()
)
forest_pivot = (
    labels.assign(forest_type=species_forest_type)
    .pivot_table(index="forest_type", columns="ogf_group", values="parcel_id", aggfunc="size")
    .reindex(index=FOREST_ORDER, columns=GROUP_COLUMNS)
    .fillna(0)
    .astype(int)
)
forest_pivot[ALL_LABEL] = forest_pivot.sum(axis=1)
print("\n[forest type: parcel counts by class]")
print(forest_pivot.to_string())

In [ ]:
import pandas as pd

from utils import terminology

FOREST_TYPES = list(terminology.CORINE_FOREST_CLASSES.values())  # broadleaf, coniferous, mixed
DISPLAY = {t: t.title() for t in FOREST_TYPES}
GROUPS = ["OGF", "Non-OGF"]

strata = labels[labels["ogf_group"].isin(GROUPS)].copy()
strata["forest_type"] = strata["class"].str.split("_").str[-1].str.lower()
strata = strata[strata["forest_type"].isin(FOREST_TYPES)]

age = (
    strata.dropna(subset=["final_age"])
    .pivot_table(index="forest_type", columns="ogf_group", values="final_age", aggfunc="median")
    .reindex(index=FOREST_TYPES, columns=GROUPS)
    .rename(index=DISPLAY)
)
print("[age: median years by class x forest type]")
print(age.round(0).to_string())

ownership = (
    strata.pivot_table(
        index=["ownership_type", "forest_type"],
        columns="ogf_group",
        values="area_ha",
        aggfunc="sum",
    )
    .reindex(columns=GROUPS)
    .fillna(0.0)
    .rename(index=DISPLAY, level="forest_type")
    .sort_values("OGF", ascending=False)
)
print("\n[ownership: area ha by class x forest type]")
print(ownership.round(0).to_string())

## 10. Forest type: species attribute versus CORINE

Compare the species-attribute forest type with each parcel's dominant CORINE forest class, by coverage and agreement.

In [ ]:
import pandas as pd

from utils import terminology

FOREST_ORDER = list(terminology.CORINE_FOREST_CLASSES.values())  # broadleaf, coniferous, mixed
GROUPS = ["OGF", "Non-OGF"]

labelled = labels[labels["ogf_group"].isin(GROUPS)].copy()
attribute = labelled["class"].str.split("_").str[-1].str.lower()
labelled["attribute_type"] = attribute.where(attribute.isin(FOREST_ORDER))
labelled["corine_type"] = labelled["corine_forest_type"]  # assigned in notebook 005

print("[coverage] parcels with a forest type assigned:")
for group in GROUPS:
    sub = labelled[labelled["ogf_group"].eq(group)]
    print(
        f"   {group}: CORINE {sub['corine_type'].notna().mean():.1%}, "
        f"species attribute {sub['attribute_type'].notna().mean():.1%}"
    )

print("\n[gap] unclassified parcels (no species attribute) that have a CORINE type:")
for group in GROUPS:
    sub = labelled[labelled["ogf_group"].eq(group)]
    no_attr = sub[sub["attribute_type"].isna()]
    print(f"   {group}: {int(no_attr['corine_type'].notna().sum())} of {len(no_attr)}")

for group in GROUPS:
    sub = labelled[labelled["ogf_group"].eq(group)].dropna(subset=["attribute_type", "corine_type"])
    agree = (sub["attribute_type"] == sub["corine_type"]).mean()
    print(f"\n[{group}] agreement on {len(sub)} parcels with both types: {agree:.1%}")
    table = pd.crosstab(sub["attribute_type"], sub["corine_type"])
    table = table.reindex(index=FOREST_ORDER, columns=FOREST_ORDER, fill_value=0)
    print(table.to_string())

### Dominance threshold sensitivity against CORINE

Vary the species dominance threshold and measure agreement between the species-attribute forest type and CORINE.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from utils import terminology
from utils.forestry import classify_forest_type
from utils.style import get_figure_size, save_figure

NOTEBOOK = "006_aoi_and_reference_label_statistics"
GROUPS = ["OGF", "Non-OGF"]
THRESHOLDS = np.arange(50, 91, 5)

both = labels[
    labels["ogf_group"].isin(GROUPS)
    & labels["corine_forest_type"].notna()
    & labels["pct_broadleaf"].notna()
].copy()


def agreement_at(frame, threshold):
    attribute = [
        classify_forest_type(b, c, dominance_pct=threshold)
        for b, c in zip(frame["pct_broadleaf"], frame["pct_coniferous"], strict=True)
    ]
    return float((np.array(attribute) == frame["corine_forest_type"].to_numpy()).mean())


curves = {"Pooled": [agreement_at(both, t) for t in THRESHOLDS]}
for group in GROUPS:
    curves[group] = [agreement_at(both[both["ogf_group"].eq(group)], t) for t in THRESHOLDS]

default = terminology.FOREST_TYPE_DOMINANCE_PCT
best = int(THRESHOLDS[int(np.argmax(curves["Pooled"]))])
at_default = curves["Pooled"][list(THRESHOLDS).index(default)]
print(f"[sensitivity] pooled agreement on {len(both):,} parcels with both labels")
print(
    f"[sensitivity] best threshold {best}% ({max(curves['Pooled']):.1%}); "
    f"current default {default}% ({at_default:.1%})"
)

fig, ax = plt.subplots(figsize=get_figure_size("single", aspect=0.78))
ax.plot(THRESHOLDS, curves["Pooled"], color="black", marker="o", linewidth=1.2, label="Pooled")
ax.plot(
    THRESHOLDS,
    curves["OGF"],
    color=terminology.SEMANTIC_COLOURS["ogf"],
    marker="o",
    label="Old-growth",
)
ax.plot(
    THRESHOLDS,
    curves["Non-OGF"],
    color=terminology.SEMANTIC_COLOURS["non_ogf"],
    marker="o",
    label="Non-old-growth",
)
ax.axvline(default, color="grey", linestyle="--", linewidth=0.8)
ax.set_xlabel("Species dominance threshold (%)")
ax.set_ylabel("Agreement with CORINE")
ax.set_ylim(0, 1)
ax.legend(fontsize=7, frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
save_figure(
    fig,
    f"{NOTEBOOK}/forest_type_dominance_sensitivity",
    data=pd.DataFrame(
        {
            "threshold_pct": THRESHOLDS,
            "pooled": curves["Pooled"],
            "ogf": curves["OGF"],
            "non_ogf": curves["Non-OGF"],
        }
    ),
)
plt.show()

## 11. Baseline predictors by class and forest type

Disaggregate the baseline predictor medians by reference class and species-attribute forest type.

In [ ]:
import pandas as pd

from utils import terminology

DISTANCE_BANDS = {"dist_paved_road_m", "dist_unpaved_road_m", "dist_footpath_m"}
BASELINE_BANDS = terminology.FEATURE_SET_BANDS["baseline"]
TYPES = ["Broadleaf", "Coniferous", "Mixed"]

stratified = parcel_stats[parcel_stats["ogf_group"].isin(["OGF", "Non-OGF"])].copy()
stratified["forest_type"] = stratified["class"].str.split("_").str[-1]
stratified = stratified[stratified["forest_type"].isin(TYPES)]
stratified["stratum"] = stratified["ogf_group"] + " " + stratified["forest_type"]
order = [f"{group} {ftype}" for group in ["OGF", "Non-OGF"] for ftype in TYPES]

print("[strata] parcels per class x forest type:")
print(stratified["stratum"].value_counts().reindex(order).to_string())

rows = {}
for band in BASELINE_BANDS:
    label = terminology.BAND_LABELS[band]
    scale = 1000.0 if band in DISTANCE_BANDS else 1.0
    if band in DISTANCE_BANDS:
        label = label.replace("(m)", "(km)")
    rows[label] = {
        stratum: f"{(group[band].dropna() / scale).median():,.1f}"
        for stratum, group in stratified.groupby("stratum")
    }

table = pd.DataFrame(rows).T.reindex(columns=order)
print("\n[strata] baseline predictor medians by class x forest type:")
print(table.to_string())

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from utils import terminology
from utils.style import get_figure_size, save_figure

NOTEBOOK = "006_aoi_and_reference_label_statistics"
GROUPS = ["OGF", "Non-OGF"]
GROUP_COLOURS = {
    "OGF": terminology.SEMANTIC_COLOURS["ogf"],
    "Non-OGF": terminology.SEMANTIC_COLOURS["non_ogf"],
}
GROUP_LABELS = {"OGF": "Old-growth", "Non-OGF": "Non-old-growth"}
FOREST_TYPES = list(terminology.CORINE_FOREST_CLASSES.values())  # broadleaf, coniferous, mixed
BASELINE_BANDS = terminology.FEATURE_SET_BANDS["baseline"]
DISTANCE_BANDS = {"dist_paved_road_m", "dist_unpaved_road_m", "dist_footpath_m"}
PANEL_TITLES = {
    "elevation_m": "Elevation (m)",
    "slope_deg": "Slope (degrees)",
    "heat_load_index": "Heat load index",
    "dist_paved_road_m": "Distance to paved\nroad (km)",
    "dist_unpaved_road_m": "Distance to unpaved\nroad (km)",
    "dist_footpath_m": "Distance to\nfootpath (km)",
    "bio01_mean_annual_temp_degC": "Mean annual\ntemperature (deg C)",
    "bio12_annual_precip_mm": "Annual precipitation\n(mm)",
}
# Two adjacent boxes (OGF, non-OGF) per forest-type group, three groups across the axis.
POSITIONS = [1.0, 1.8, 3.2, 4.0, 5.4, 6.2]
GROUP_CENTRES = [1.4, 3.6, 5.8]

stratified = parcel_stats[parcel_stats["ogf_group"].isin(GROUPS)].copy()
stratified["forest_type"] = stratified["class"].str.split("_").str[-1].str.lower()
stratified = stratified[stratified["forest_type"].isin(FOREST_TYPES)]

n_by_group = {}
for group in GROUPS:
    finite = stratified.loc[stratified["ogf_group"].eq(group), BASELINE_BANDS[0]]
    n_by_group[group] = int(finite.notna().sum())

with plt.rc_context(
    {
        "font.size": 7,
        "axes.titlesize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
    }
):
    fig, axes = plt.subplots(2, 3, figsize=get_figure_size("double", aspect=0.62))
    for ax, band in zip(axes.flat, BASELINE_BANDS, strict=True):
        scale = 1000.0 if band in DISTANCE_BANDS else 1.0
        data = []
        colours = []
        for ftype in FOREST_TYPES:
            for group in GROUPS:
                mask = stratified["forest_type"].eq(ftype) & stratified["ogf_group"].eq(group)
                data.append(stratified.loc[mask, band].dropna().to_numpy() / scale)
                colours.append(GROUP_COLOURS[group])
        box = ax.boxplot(
            data,
            positions=POSITIONS,
            widths=0.7,
            patch_artist=True,
            showfliers=False,
            medianprops=dict(color="black", linewidth=1.0),
        )
        for patch, colour in zip(box["boxes"], colours, strict=True):
            patch.set_facecolor(colour)
            patch.set_alpha(0.85)
        ax.set_title(PANEL_TITLES[band], pad=3)
        ax.set_xticks(GROUP_CENTRES)
        ax.set_xticklabels([t.title() for t in FOREST_TYPES])
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.margins(y=0.05)

    handles = [
        Patch(
            facecolor=GROUP_COLOURS[group],
            edgecolor="black",
            alpha=0.85,
            label=f"{GROUP_LABELS[group]} (n = {n_by_group[group]:,})",
        )
        for group in GROUPS
    ]
    fig.legend(
        handles=handles, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.93)
    )
    fig.suptitle("Baseline predictors by reference class and forest type", y=0.99, fontsize=9)
    fig.subplots_adjust(left=0.07, right=0.99, top=0.83, bottom=0.13, wspace=0.42, hspace=0.68)

save_figure(
    fig,
    f"{NOTEBOOK}/baseline_predictors_by_class_and_forest_type",
    data=stratified[["ogf_group", "forest_type", *BASELINE_BANDS]],
)
plt.show()

### By CORINE forest type (manuscript figure)

Repeat the previous figure with each parcel's forest type taken from its dominant CORINE forest class rather than the species attribute, so that every parcel with a CORINE type is stratified (including those the species attribute leaves unclassified).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from utils import terminology
from utils.style import get_figure_size, save_figure

NOTEBOOK = "006_aoi_and_reference_label_statistics"
GROUPS = ["OGF", "Non-OGF"]
GROUP_COLOURS = {
    "OGF": terminology.SEMANTIC_COLOURS["ogf"],
    "Non-OGF": terminology.SEMANTIC_COLOURS["non_ogf"],
}
GROUP_LABELS = {"OGF": "Old-growth", "Non-OGF": "Non-old-growth"}
FOREST_TYPES = list(terminology.CORINE_FOREST_CLASSES.values())  # broadleaf, coniferous, mixed
BASELINE_BANDS = terminology.FEATURE_SET_BANDS["baseline"]
DISTANCE_BANDS = {"dist_paved_road_m", "dist_unpaved_road_m", "dist_footpath_m"}
PANEL_TITLES = {
    "elevation_m": "Elevation (m)",
    "slope_deg": "Slope (degrees)",
    "heat_load_index": "Heat load index",
    "dist_paved_road_m": "Distance to paved\nroad (km)",
    "dist_unpaved_road_m": "Distance to unpaved\nroad (km)",
    "dist_footpath_m": "Distance to\nfootpath (km)",
    "bio01_mean_annual_temp_degC": "Mean annual\ntemperature (deg C)",
    "bio12_annual_precip_mm": "Annual precipitation\n(mm)",
}
# Two adjacent boxes (OGF, non-OGF) per forest-type group, three groups across the axis.
POSITIONS = [1.0, 1.8, 3.2, 4.0, 5.4, 6.2]
GROUP_CENTRES = [1.4, 3.6, 5.8]

# Forest type from each parcel's dominant CORINE forest class (assigned in notebook 005),
# which covers parcels the species attribute leaves unclassified.
stratified = parcel_stats[parcel_stats["ogf_group"].isin(GROUPS)].copy()
stratified["forest_type"] = stratified["corine_forest_type"].str.lower()
stratified = stratified[stratified["forest_type"].isin(FOREST_TYPES)]

n_by_group = {}
for group in GROUPS:
    finite = stratified.loc[stratified["ogf_group"].eq(group), BASELINE_BANDS[0]]
    n_by_group[group] = int(finite.notna().sum())

print("[strata] parcels per class x CORINE forest type:")
print(
    stratified.pivot_table(
        index="forest_type", columns="ogf_group", values="parcel_id", aggfunc="size"
    )
    .reindex(index=FOREST_TYPES, columns=GROUPS)
    .to_string()
)

with plt.rc_context(
    {
        "font.size": 7,
        "axes.titlesize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
    }
):
    fig, axes = plt.subplots(2, 3, figsize=get_figure_size("double", aspect=0.62))
    for ax, band in zip(axes.flat, BASELINE_BANDS, strict=True):
        scale = 1000.0 if band in DISTANCE_BANDS else 1.0
        data = []
        colours = []
        for ftype in FOREST_TYPES:
            for group in GROUPS:
                mask = stratified["forest_type"].eq(ftype) & stratified["ogf_group"].eq(group)
                data.append(stratified.loc[mask, band].dropna().to_numpy() / scale)
                colours.append(GROUP_COLOURS[group])
        box = ax.boxplot(
            data,
            positions=POSITIONS,
            widths=0.7,
            patch_artist=True,
            showfliers=False,
            medianprops=dict(color="black", linewidth=1.0),
        )
        for patch, colour in zip(box["boxes"], colours, strict=True):
            patch.set_facecolor(colour)
            patch.set_alpha(0.85)
        ax.set_title(PANEL_TITLES[band], pad=3)
        ax.set_xticks(GROUP_CENTRES)
        ax.set_xticklabels([t.title() for t in FOREST_TYPES])
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.margins(y=0.05)

    handles = [
        Patch(
            facecolor=GROUP_COLOURS[group],
            edgecolor="black",
            alpha=0.85,
            label=f"{GROUP_LABELS[group]} (n = {n_by_group[group]:,})",
        )
        for group in GROUPS
    ]
    fig.legend(
        handles=handles, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.93)
    )
    fig.suptitle(
        "Baseline predictors by reference class and CORINE forest type", y=0.99, fontsize=9
    )
    fig.subplots_adjust(left=0.07, right=0.99, top=0.83, bottom=0.13, wspace=0.42, hspace=0.68)

save_figure(
    fig,
    f"{NOTEBOOK}/fig_s1_baseline_predictors_by_class_and_corine_forest_type",
    data=stratified[["ogf_group", "forest_type", *BASELINE_BANDS]],
)
plt.show()